# 思维链推理 (Chain-of-Thought)

> **学习目标**：掌握 CoT 推理技术，提升模型复杂推理能力

---

## 目录

1. [CoT 概述](#1-cot-概述)
2. [Zero-shot CoT](#2-zero-shot-cot)
3. [Few-shot CoT](#3-few-shot-cot)
4. [Self-Consistency](#4-self-consistency)
5. [高级策略](#5-高级策略)

In [ ]:
import sys
sys.path.append('..')

from src.chain_of_thought import (
    CoTPrompt, CoTExample, CoTExamples,
    SelfConsistency, TreeOfThought
)

## 1. CoT 概述

**Chain-of-Thought (CoT)**：让模型展示推理过程，而不是直接给出答案。

```
标准提示：问题 → 答案
CoT提示：问题 → 步骤1 → 步骤2 → ... → 答案
```

In [ ]:
# CoT vs 标准提示对比

standard_prompt = """问题：一个商店有15个苹果，卖出了8个，又进货了12个，现在有多少个苹果？

答案："""

cot_prompt = """问题：一个商店有15个苹果，卖出了8个，又进货了12个，现在有多少个苹果？

让我们一步一步思考：
1. 初始苹果数量：15个
2. 卖出后剩余：15 - 8 = 7个
3. 进货后总数：7 + 12 = 19个

答案：19个苹果"""

print("=== 标准提示 ===")
print(standard_prompt)
print("\n=== CoT 提示 ===")
print(cot_prompt)

## 2. Zero-shot CoT

只需添加魔法咒语：**"Let's think step by step"**

In [ ]:
# Zero-shot CoT
cot = CoTPrompt(strategy="zero_shot_cot", language="zh")

questions = [
    "如果小明有5个苹果，给了小红2个，又买了3个，现在有几个？",
    "一辆车以60km/h的速度行驶，2小时能走多远？",
    "如果今天是周三，那么10天后是周几？",
]

for q in questions:
    print(f"问题：{q}")
    print(cot.format(q))
    print("-" * 50)

In [ ]:
# 不同语言的触发词
print("=== 中文触发词 ===")
cot_zh = CoTPrompt(strategy="zero_shot_cot", language="zh")
print(cot_zh.format("计算 2+3×4"))

print("\n=== 英文触发词 ===")
cot_en = CoTPrompt(strategy="zero_shot_cot", language="en")
print(cot_en.format("Calculate 2+3×4"))

## 3. Few-shot CoT

提供带推理过程的示例。

In [ ]:
# 使用预定义的数学示例
cot_few = CoTPrompt(
    strategy="few_shot_cot",
    examples=CoTExamples.MATH_EXAMPLES
)

print(cot_few.format("小红有10元钱，买了3元的笔，又买了2元的本子，还剩多少钱？"))

In [ ]:
# 自定义 CoT 示例
logic_examples = [
    CoTExample(
        question="所有的猫都是动物，小花是一只猫，小花是动物吗？",
        reasoning="""1. 前提1：所有的猫都是动物
2. 前提2：小花是一只猫
3. 根据前提1，猫属于动物类别
4. 根据前提2，小花属于猫类别
5. 因此，小花属于动物类别""",
        answer="是的，小花是动物"
    ),
]

cot_logic = CoTPrompt(strategy="few_shot_cot", examples=logic_examples)
print(cot_logic.format("所有的鸟都会飞，企鹅是鸟，企鹅会飞吗？"))

## 4. Self-Consistency

多次采样，投票选择最一致的答案。

In [ ]:
# Self-Consistency 示例
sc = SelfConsistency(n_samples=5, temperature=0.7)

# 模拟生成函数
import random
def mock_generate(prompt, temperature):
    # 模拟不同的推理路径
    responses = [
        "让我计算一下...\n15 - 8 = 7\n7 + 12 = 19\n答案：19个",
        "首先卖出8个：15-8=7\n然后进货12个：7+12=19\n答案：19个",
        "初始15个，卖8个剩7个，进12个共19个\n答案：19个",
        "15 - 8 + 12 = 19\n答案：19个",
        "计算：15-8=7, 7+12=19\n答案：19个",
    ]
    return random.choice(responses)

prompt = "一个商店有15个苹果，卖出了8个，又进货了12个，现在有多少个苹果？"
result = sc.generate(prompt, mock_generate)

print(f"最终答案：{result['answer']}")
print(f"置信度：{result['confidence']:.0%}")
print(f"答案分布：{result['answer_distribution']}")

## 5. 高级策略

### 5.1 Plan-and-Solve

In [ ]:
# Plan-and-Solve 策略
cot_plan = CoTPrompt(strategy="plan_and_solve", language="zh")
print(cot_plan.format("计算 (2+3) × (4+5) - 10 的结果"))

In [ ]:
# 5.2 自定义触发词
custom_cot = CoTPrompt(
    strategy="zero_shot_cot",
    custom_trigger="让我们仔细分析这个问题，逐步推导："
)
print(custom_cot.format("如果A>B，B>C，那么A和C的关系是什么？"))

### 5.3 CoT 效果对比

| 任务类型 | 标准Prompting | CoT Prompting | 提升 |
|---------|--------------|---------------|------|
| 算术推理 | 17.7% | 78.7% | +61% |
| 常识推理 | 52.4% | 73.5% | +21% |
| 符号推理 | 13.8% | 99.6% | +86% |

*数据来源：Wei et al. (2022)*

## 总结

1. **Zero-shot CoT**：添加"让我们一步一步思考"
2. **Few-shot CoT**：提供带推理过程的示例
3. **Self-Consistency**：多次采样投票
4. **适用场景**：数学推理、逻辑推理、多步骤任务

下一节：**提示优化技术**